In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
# Requires transformers>=4.51.0
# Requires sentence-transformers>=2.7.0

from sentence_transformers import SentenceTransformer

# Load the model
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

# We recommend enabling flash_attention_2 for better acceleration and memory saving,
# together with setting `padding_side` to "left":
# model = SentenceTransformer(
#     "Qwen/Qwen3-Embedding-4B",
#     model_kwargs={"attn_implementation": "flash_attention_2", "device_map": "auto"},
#     tokenizer_kwargs={"padding_side": "left"},
# )

# The queries and documents to embed


In [ ]:
model.prompts

query = "Instruct: Given a search query (could be question, title or text), retrieve relevant passages that answer / describe the query\nQuery:"

{'query': 'Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:',
 'document': ''}

In [ ]:
queries = [
    "What is the capital of China?",
    "Explain gravity",
]
documents = [
    "The capital of China is Beijing.",
    "Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun.",
]

# Encode the queries and documents. Note that queries benefit from using a prompt
# Here we use the prompt called "query" stored under `model.prompts`, but you can
# also pass your own prompt via the `prompt` argument
query_embeddings = model.encode(queries, prompt_name="query")
document_embeddings = model.encode(documents)

# Compute the (cosine) similarity between the query and document embeddings
similarity = model.similarity(query_embeddings, document_embeddings)
print(similarity)
# tensor([[0.7493, 0.0751],
#         [0.0880, 0.6318]])


tensor([[0.8972, 0.1321],
        [0.1833, 0.7466]])


In [7]:
queries = [
    "What is the capital of China?",
    "Explain gravity",
]

query_embeddings1 = model.encode(queries, prompt_name="query")
query_embeddings2 = model.encode(queries)

# Compute the (cosine) similarity between the query and document embeddings
similarity = model.similarity(query_embeddings1, query_embeddings2)
print(similarity)
# tensor([[0.7493, 0.0751],
#         [0.0880, 0.6318]])


tensor([[0.7851, 0.1430],
        [0.1927, 0.8216]])


In [3]:
# transformning beir eval format

import json
import os
path = "/rhome/sawale/indus_traning/sentense_transformers/eval/results_json/beir_eval_results/"


output = {}

# loop through each file in the directory
for file_name in os.listdir(path):
    
    if file_name.endswith(".json"):
        with open(os.path.join(path, file_name), 'r') as file:
            data = json.load(file)

        for m, v1 in data.items():
            if m not in output:
                output[m] = {}

            for metric, value in v1.items():
                subset = "_".join(metric.split("_")[:-3])
                remaining_metric = "_".join(metric.split("_")[-2:])
                new_metric_name = f"beir__{subset}__evaluator_{remaining_metric}"
                output[m][new_metric_name] = value
        

output

{'modernbert-embed-base': {'beir__scidocs__evaluator_cosine_accuracy@1': 0.229,
  'beir__scidocs__evaluator_cosine_accuracy@3': 0.41,
  'beir__scidocs__evaluator_cosine_accuracy@5': 0.49,
  'beir__scidocs__evaluator_cosine_accuracy@10': 0.59,
  'beir__scidocs__evaluator_cosine_precision@1': 0.229,
  'beir__scidocs__evaluator_cosine_precision@3': 0.17833333333333332,
  'beir__scidocs__evaluator_cosine_precision@5': 0.146,
  'beir__scidocs__evaluator_cosine_precision@10': 0.1039,
  'beir__scidocs__evaluator_cosine_recall@1': 0.046483333333333335,
  'beir__scidocs__evaluator_cosine_recall@3': 0.10843333333333333,
  'beir__scidocs__evaluator_cosine_recall@5': 0.14798333333333336,
  'beir__scidocs__evaluator_cosine_recall@10': 0.2104666666666667,
  'beir__scidocs__evaluator_cosine_ndcg@1': 0.229,
  'beir__scidocs__evaluator_cosine_ndcg@3': 0.19020124687659165,
  'beir__scidocs__evaluator_cosine_ndcg@5': 0.1661651146011697,
  'beir__scidocs__evaluator_cosine_ndcg@10': 0.20006709016585228,
  